In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: C:\Users\Aislay\Desktop\FRE 9743 Special Topics in Risk Management\Assignments\Homework 1\FRE-GY-9743-Assignments-1
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Implementation notes

The implementation is in `fixedincomelib/utilities/numerics.py`.
Use nonempty, finite, strictly increasing knots and matching finite ordinates.
Queries and integration limits in this exercise are finite real numbers.

For `axis1 = [1, 3, 5, 7]` and `values = [3, 4, 5, 6]`,

$$
f(x)=\begin{cases}
3,&x\leq 1,\\
4,&1<x\leq 3,\\
5,&3<x\leq 5,\\
6,&x>5.
\end{cases}
$$

- **Interpolation:** `searchsorted(..., side="left")` selects the first knot
  greater than or equal to the query. Capping the index at the last ordinate
  gives flat right extrapolation; queries on the left select the first ordinate.
- **Value gradient:** only the selected ordinate affects the result, so its
  derivative is one and all other derivatives are zero. These derivatives are
  with respect to `values`, not the query or the knot positions.
- **Integral gradient:** let $w_i$ be the overlap length between the integration
  interval and the region using ordinate $y_i$. Then
  $I=\sum_i w_i y_i$ and $\partial I/\partial y_i=w_i$.
  Reverse limits negate every weight; equal limits give zero weights.
- **Integration:** take the dot product of the signed weights and ordinates.
- **Bump and revalue:** approximate each derivative by
  $[I(y+\varepsilon e_i)-I(y)]/\varepsilon$, rebuilding the interpolator after
  each independent bump. The input values are copied rather than modified.

For example, over $[0.5,3.2]$ the weights are $[0.5,2,0.2,0]$, so
$I=0.5(3)+2(4)+0.2(5)=10.5$.
The final two original regions $(5,7]$ and $(7,\infty)$ use the same value and
are merged for integration. Individual endpoint values do not affect area.


### Test interpolation

In [2]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3.0, expected 3.0, diff = 0.0
f(1.0) = 3.0, expected 3.0, diff = 0.0
f(1.5) = 4.0, expected 4.0, diff = 0.0
f(3.0) = 4.0, expected 4.0, diff = 0.0
f(5.5) = 6.0, expected 6.0, diff = 0.0
f(6.5) = 6.0, expected 6.0, diff = 0.0
f(8.0) = 6.0, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [3]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [4]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s: float,
    x_e: float,
    axis1: List,
    values: List,
    interp_method: str,
    extrap_method: str,
    bump_size: Optional[float] = 1e-4
):
    """Estimate the integral gradient by bumping one ordinate at a time."""
    if bump_size is None or not np.isfinite(bump_size) or bump_size <= 0:
        raise ValueError("bump_size must be finite and positive")

    # Copies avoid changing the caller's values or accumulating bumps.
    base_values = np.array(values, dtype=float, copy=True)
    base_interpolator = qfCreate1DInterpolator(
        axis1, base_values, interp_method, extrap_method
    )
    base_integral = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = np.zeros(len(base_values), dtype=float)
    for i in range(len(base_values)):
        bumped_values = base_values.copy()
        bumped_values[i] += bump_size
        bumped_interpolator = qfCreate1DInterpolator(
            axis1, bumped_values, interp_method, extrap_method
        )
        bumped_integral = qfInterpolate1DIntegral(
            x_s, x_e, bumped_interpolator
        )
        grad[i] = (bumped_integral - base_integral) / bump_size

    return grad


### Interpolation sensitivity

In [5]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [6]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 1.659827830735594e-11
[1.5, 5.2]: max abs diff = 2.1259438653942198e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 7.571543392259628e-11


### Additional validation

The checks below assert the supplied results, compare both gradients with finite differences, and exercise boundary cases and irregular grids. Small floating-point differences are expected; an assertion failure indicates a discrepancy that needs investigation.

In [7]:
# Check the instructor's expected values with assertions, not only prints.
value_errors = []
integral_errors = []
for x, expected in test_points:
    np.testing.assert_allclose(qfInterpolate1D(x, interp_1d), expected,
                               rtol=0, atol=1e-12)
    analytic = qfInterpolate1DGrad(x, interp_1d)
    numeric = bump_reval_interpolator(
        x, axis1, values, interp_method, extrap_method
    )
    np.testing.assert_allclose(analytic, numeric, rtol=1e-8, atol=1e-8)
    value_errors.append(np.max(np.abs(analytic - numeric)))

for (a, b), expected in integration_cases:
    np.testing.assert_allclose(qfInterpolate1DIntegral(a, b, interp_1d),
                               expected, rtol=0, atol=1e-12)
    analytic = qfInterpolate1DIntegralGrad(a, b, interp_1d)
    original_values = np.array(values, dtype=float, copy=True)
    numeric = bump_reval_interpolator_integrand(
        a, b, axis1, values, interp_method, extrap_method
    )
    np.testing.assert_array_equal(values, original_values)
    np.testing.assert_allclose(analytic, numeric, rtol=1e-8, atol=1e-8)
    np.testing.assert_allclose(np.sum(analytic), b - a, rtol=0, atol=1e-12)
    integral_errors.append(np.max(np.abs(analytic - numeric)))

# Exactly at a node and immediately to either side.
for i, knot in enumerate(axis1):
    np.testing.assert_allclose(qfInterpolate1D(knot, interp_1d), values[i])
    below = np.nextafter(float(knot), -np.inf)
    above = np.nextafter(float(knot), np.inf)
    np.testing.assert_allclose(qfInterpolate1D(below, interp_1d), values[i])
    np.testing.assert_allclose(qfInterpolate1D(above, interp_1d),
                               values[min(i + 1, len(values) - 1)])

# Zero-width intervals, reversed limits, and a single-node interpolator.
np.testing.assert_allclose(qfInterpolate1DIntegral(2, 2, interp_1d), 0)
np.testing.assert_array_equal(qfInterpolate1DIntegralGrad(2, 2, interp_1d),
                              np.zeros(len(values)))
np.testing.assert_allclose(qfInterpolate1DIntegral(3.2, 0.5, interp_1d), -10.5)
np.testing.assert_allclose(qfInterpolate1DIntegralGrad(3.2, 0.5, interp_1d),
                           [-0.5, -2.0, -0.2, 0.0])
single = qfCreate1DInterpolator([2.0], [7.0], interp_method, extrap_method)
np.testing.assert_allclose(qfInterpolate1D(-10, single), 7)
np.testing.assert_allclose(qfInterpolate1D(10, single), 7)
np.testing.assert_allclose(qfInterpolate1DIntegral(-3, 5, single), 56)
np.testing.assert_allclose(qfInterpolate1DIntegralGrad(-3, 5, single), [8])

# Independent area check on irregular knots and signed ordinate values.
# Splitting at knots makes midpoint rectangle integration exact here.
rng = np.random.default_rng(9743)
for _ in range(30):
    knots = np.cumsum(rng.uniform(0.1, 2.0, size=5))
    heights = rng.normal(size=5)
    candidate = qfCreate1DInterpolator(knots, heights,
                                      interp_method, extrap_method)
    a, b = sorted(rng.uniform(knots[0] - 2, knots[-1] + 2, size=2))
    cuts = np.r_[a, knots[(knots > a) & (knots < b)], b]
    expected = sum((right - left) * qfInterpolate1D((left + right) / 2, candidate)
                   for left, right in zip(cuts[:-1], cuts[1:]))
    np.testing.assert_allclose(qfInterpolate1DIntegral(a, b, candidate), expected,
                               rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(
        qfInterpolate1DIntegralGrad(a, b, candidate),
        bump_reval_interpolator_integrand(a, b, knots, heights,
                                         interp_method, extrap_method),
        rtol=1e-8, atol=1e-8
    )

print(f"Passed {len(test_points)} interpolation examples and "
      f"{len(integration_cases)} integration examples.")
print(f"Maximum value-gradient error: {max(value_errors):.3e}")
print(f"Maximum integral-gradient error: {max(integral_errors):.3e}")
print("Passed node-boundary, reversed-limit, zero-width, single-node, "
      "input-preservation and 30 irregular-grid checks.")


Passed 7 interpolation examples and 8 integration examples.
Maximum value-gradient error: 2.331e-12
Maximum integral-gradient error: 7.572e-11
Passed node-boundary, reversed-limit, zero-width, single-node, input-preservation and 30 irregular-grid checks.
